## 物件導向程式設計 (OOP) 與 Pydantic 資料模型

- 目標：掌握 Python 物件導向程式設計（OOP）的**繼承與多態**，並能清楚**分辨 Dataclass 與 Pydantic 的底層差異**與半導體專案中的使用時機。


### 1. 記憶體中資料結構型態定義 (typing)

- 概念：
    - 型態提示不是強制類型其他檢查：Python 執行期不會因為型態不符而報錯，型態提示主要提供 IDE 自動補全、靜態分析工具（如 mypy）以及協作者閱讀用。
    - 使用型態：
        - List[int]、Dict[str, float]、Tuple[int, int]：容器內部元素型態標誌。
        - Optional[X]：代表 Union[X, None]，常用於函數參數可選（Optional）或未提供的情況，例如尚未讀取到的機台狀態。
        - Union[X, Y]：值可能是多種類型之一，例如解析出來的欄位值可能是 str 或 float。
        - Callable[[int, int], bool]：標註一個函數參數本身的簽章（輸入型態與回傳型態）。
        - TypeVar：用來寫泛型函數，例如一個「無論傳進來是 List 還是 Dict 都會回傳同型態」的工具函數。
    - Python 3.9+以後的新寫法：可以直接用內建型態 list[int]、dict[str, float] 取代 typing.List，typing.Dict 語法更簡潔（但若要支援舊版 Python 仍需 from typing import ...）。
    - 為了讓大型自動化程式碼更容易維護、減少型態錯誤，並在 IDE（如 VS Code / PyCharm）中提供自動補全，會廣泛使用型態提示。


### 2. 類別繼承與多態

- 概念：
    - 繼承：子類別自動取得父類別的屬性與方法，用 class Child(Parent): 聲明；適合用在「不同品牌指示燈卡控制器佔用邏輯大部分，只有連線指令不同」的場景。
    - 多態（Polymorphism）：不同的子類別可以對同一個方法名稱有各自的實作，呼叫端只需要面對統一介面（如 controller.connect()），不需要知道底層是哪個廠牌。
    - 抽象類別（ABC）：貫穿 abc.ABC 與 @abstractmethod 強制規定子類別「一定要實作」某些方法（如connect()、read_status()），否則工件連建立都會失敗，可提早在開發期間抓到漏寫方法的錯誤，而不是等到產線執行才出錯。
    - super()：子類別覆寫方法時，仍可透過 super().init() 呼叫父類別的初始化邏輯，以達到程式碼複用並避免重複編寫。
    - 組合 vs 繼承：當不同機台之間的差異變大、硬套繼承邏輯顯得牽強時，使用「組合（Composition，將功能物件封裝為類別的屬性）」通常比硬用繼承更具彈性，這是實務上常見的設計選擇問題。


- 實作：
    - 在測試產線中，我們需要控制不同廠牌或型號的探針卡控制器（Probe Card Controller）。為了**讓主程式維持統一的介面**，我們定義一個基底類別（Base），再由**各個型號去繼承並實作各自的控制邏輯（多態性）**。


In [ ]:
from abc import ABC, abstractmethod
import time


class BaseProbeCardController(ABC):
    """探針卡控制器抽象基底類別"""

    def __init__(self, ip_address: str, port: int):
        self.ip_address = ip_address
        self.port = port
        self.is_connected = False

    @abstractmethod
    def connect(self) -> bool:
        """所有控制器都必須實作的連線邏輯"""
        pass

    @abstractmethod
    def move_to_die(self, x: int, y: int) -> bool:
        """移動到特定晶粒 (Die) 位置"""
        pass

    def disconnect(self):
        """共用的斷線邏輯"""
        if self.is_connected:
            print(f"[Base] 已中斷與 {self.ip_address} 的連線。")
            self.is_connected = False


class AdvantestProbeController(BaseProbeCardController):
    """愛德萬型號探針卡控制器"""

    def connect(self) -> bool:
        print(f"[Advantest] 正在透過 TCP/IP 連線至 {self.ip_address}:{self.port}...")
        time.sleep(0.2)  # 模擬硬體交握
        self.is_connected = True
        return True

    def move_to_die(self, x: int, y: int) -> bool:
        if not self.is_connected:
            raise RuntimeError("機台未連線，無法移動！")
        print(f"[Advantest] 成功移動針床至晶圓座標 (X: {x}, Y: {y})")
        return True


class TeradyneProbeController(BaseProbeCardController):
    """泰瑞達型號探針卡控制器"""

    def connect(self) -> bool:
        print(f"[Teradyne] 啟動專有通訊協定，連線至 {self.ip_address}...")
        self.is_connected = True
        return True

    def move_to_die(self, x: int, y: int) -> bool:
        if not self.is_connected:
            raise RuntimeError("機台未連線，無法移動！")
        # 泰瑞達機台內部可能需要暫存區或特殊補償值
        print(f"[Teradyne] 套用精度補償，移動至物理座標 (X: {x}, Y: {y})")
        return True


# -- 多態性 (Polymorphism) 實戰測試 --
def execute_test_sequence(controller: BaseProbeCardController):
    """主程式不需要知道是哪家廠牌，只要符合基底介面就能執行"""
    controller.connect()
    controller.move_to_die(12, 45)
    controller.disconnect()
    print("-" * 40)


# 同一套流水線，丟入不同機台物件
print(">>> 測試愛德萬控制器：")
execute_test_sequence(AdvantestProbeController("192.168.1.100", 5025))

print(">>> 測試泰瑞達控制器：")
execute_test_sequence(TeradyneProbeController("192.168.1.101", 8080))

### 3. Dataclass vs Pydantic BaseModel 差異與使用時機

- 概念：
    - 晶圓測試完成後會產生**點位資料（Wafer Bin Map）與追蹤日誌**。我們需要將這些資料封裝成物件。究竟該用 Python 原生的 dataclasses 還是一方之霸 pydantic？


#### 3.1 Python 原生 Dataclass（適合**內部資料封裝、計算**）

- 優點：效能極快、內建不需額外安裝套件。
- 缺點：只會做「型別提示」，完全不具備底層資料型別強制驗證能力。


In [ ]:
from dataclasses import dataclass
from datetime import datetime


@dataclass
class LogEntryDataclass:
    timestamp: datetime
    lot_id: str
    wafer_no: int
    yield_rate: float


# --- 測試 Dataclass ---
# 故意塞入錯的型別（例如：將 wafer_no 塞入字串 "Third"，yield_rate 塞入 "100%"）
wrong_data_dc = LogEntryDataclass(
    timestamp=datetime.now(),
    lot_id="LOT_A001",
    wafer_no="Third",  # 型別錯誤！應該要是 int
    yield_rate="100%",  # 型別錯誤！應該要是 float
)

print("Dataclass 驗證缺陷演示：")
print(wrong_data_dc)
# 結果：它直接接受了錯誤的字串，並不會報錯！這在測試資料寫入資料庫時會引發系統崩潰。

#### 3.2 Pydantic BaseModel（適合**對外介面、資料清洗與驗證**）

- 優點：嚴格執行資料型別轉換與欄位邏輯驗證，能把進來的髒資料（如字串數字）自動轉成正確型別。
- 缺點：因為有驗證邏輯，運行速度稍微比 Dataclass 慢。


In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import List


class WaferBinMapPydantic(BaseModel):
    lot_id: str
    wafer_no: int = Field(..., ge=1, le=25)  # 限制晶圓片號必須在 1~25 片之間
    bin_counts: List[int]  # 每個 Bin 數量的陣列

    @field_validator("lot_id")
    @classmethod
    def validate_lot_id(cls, v: str) -> str:
        """自訂驗證：Lot ID 必須以 LOT 關鍵字開頭"""
        if not v.startswith("LOT_"):
            raise ValueError("Lot ID 命名不符合半導體規範，必須以 'LOT_' 開頭")
        return v


# ---- 測試 Pydantic 成功轉換（自動轉型） ----
print("\n Pydantic 自動轉型演示：")
clean_data = WaferBinMapPydantic(
    lot_id="LOT_B999",
    wafer_no="12",  # 故意輸入字串型態的 "12"
    bin_counts=[12000, 45, 12],
)
print(
    f"自動轉型後的 wafer_no 型別: {type(clean_data.wafer_no)}, 數值: {clean_data.wafer_no}"
)

# ---- 測試 Pydantic 驗證失敗攔截 ----
print("\n Pydantic 錯誤攔截演示：")
try:
    invalid_data = WaferBinMapPydantic(
        lot_id="BAD_ID_001",  # 觸發 validator 錯誤
        wafer_no=99,  # 觸發 le=25 範圍錯誤
        bin_counts=[100],
    )
except Exception as e:
    print(e)

- 總結：在我的專案中，我設計了 ProbeCardController 的繼承架構，讓主控制程序不需要更動就能抽換不同的測試機台。而在資料處理模組中，我會將內部需要頻繁高計算量的資料封裝成 Dataclass 以追求效能；但對於從 MES Log 清洗出來、或是要寫入資料庫的 WaferBinMap，我一定採用 Pydantic。這樣可以在資料流的最上游就擋掉格式錯誤與超出 1~25 片範圍的髒資料，確保系統穩定性。
